# CRF HDP-GMM five-seed metrics

This notebook is based on `CRF_hdp_gmm.ipynb`. It runs the collapsed Gibbs sampler with the CRF scheme five times under different seeds and reports mean and sample standard deviation for ACC, DDA, Brier score, ECE, and runtime.


In [1]:
import numpy as np
import ray
from copy import deepcopy
import matplotlib.pyplot as plt
import scipy.special as ssp

from hdpgmm_syn import Gaussian, WorkerLevelGibbsSampler
from hdpgmm import GibbsSampler
from model_loglikelihood import dict2mix, all_loglike
import time
import warnings
warnings.filterwarnings("ignore", category=RuntimeWarning)

import pandas as pd
import torch
from scipy.optimize import linear_sum_assignment
from typing import List, Callable, Union, Any, TypeVar, Tuple
Tensor = TypeVar('torch.tensor')

from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
import matplotlib

from collections import Counter

seed = 42
torch.manual_seed(seed)  # ensure reproducible results
np.random.seed(seed)


e:\Anaconda\envs\vscode\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2026-07-20 13:40:25,836	INFO util.py:154 -- Missing packages: ['ipywidgets']. Run `pip install -U ipywidgets`, then restart the notebook server for rich notebook output.


# Dataset


In [ ]:
'''use a numerical dataset to test the model
label  damaged_floor     damage_extent         number_of_samples
0           0            0%                     1500
1           1            3%                     500
2           1            6%                     500
3           1            10%                    500
4           1,3          5%,10%                 500
5           1,3,5        5%,10%,15%             500
6           2,4,6        10%,15%,20%            500
7           1,3,5,7      10%,15%,20%,25%        500


'''
# Read data from CSV
features = pd.read_csv("TF_mag_numerical_8class_1000features_20dB.csv")
features = features.astype("float32")
# # # Convert DataFrame to PyTorch tensors
X = torch.tensor(features.values[:,:])
X = X.t()

input_dim = X.shape[1]
print(X.shape)

a = []
for i in range(1500):
    a.append(0)
for i in np.arange(1500,2000):
    a.append(1)
for i in np.arange(2000,2500):
    a.append(2)
for i in np.arange(2500,3000):
    a.append(3)
for i in np.arange(3000,3500):
    a.append(4)
for i in np.arange(3500,4000):
    a.append(5)
for i in np.arange(4000,4500):
    a.append(6)
for i in np.arange(4500,5000):
    a.append(7)
print(len(a))
y0 = torch.tensor(a)
y0 = y0.unsqueeze(1)
y = [int(_) for _ in y0]
y = torch.tensor(y)
num_classes = y.max().item() + 1
print(f"number of total classes: {num_classes}")

plt.plot(y)
plt.show()


# Brier score


In [4]:
import numpy as np
from scipy.special import logsumexp

# -------------------------- core metrics -------------------------- #
def brier_score(p_hat, y_true, classes=None, sample_weight=None):
    """
    Multiclass Brier score:
      BS = (1/N) * sum_i sum_c (p_hat[i,c] - 1{y_i=c})^2
    """
    p_hat = np.asarray(p_hat, dtype=float)
    N, C = p_hat.shape
    if classes is None:
        classes = np.unique(y_true)
    class_to_idx = {c:i for i,c in enumerate(classes)}
    y_idx = np.array([class_to_idx[y] for y in y_true], dtype=int)
    Y = np.eye(C)[y_idx]   # one-hot
    if sample_weight is None:
        return np.mean(np.sum((p_hat - Y)**2, axis=1))
    w = np.asarray(sample_weight, dtype=float)
    w /= w.sum()
    return np.sum(w * np.sum((p_hat - Y)**2, axis=1))

def brier_by_class(p_hat, y_true, classes=None):
    """Classwise Brier (averaged over samples of that class)."""
    p_hat = np.asarray(p_hat, dtype=float)
    if classes is None:
        classes = np.unique(y_true)
    C = len(classes)
    class_to_idx = {c:i for i,c in enumerate(classes)}
    y_idx = np.array([class_to_idx[y] for y in y_true], dtype=int)
    Y = np.eye(C)[y_idx]
    out = {}
    for c, cl in enumerate(classes):
        mask = (y_idx == c)
        if mask.any():
            out[cl] = np.mean(np.sum((p_hat[mask] - Y[mask])**2, axis=1))
        else:
            out[cl] = np.nan
    return out

def brier_skill_score(p_hat, y_true, classes=None):
    """
    BSS = 1 - BS / BS_ref, where BS_ref uses climatology (class frequency).
    """
    if classes is None:
        classes = np.unique(y_true)
    C = len(classes)
    # climatology
    counts = np.array([(y_true == c).sum() for c in classes], dtype=float)
    p_ref = counts / counts.sum()
    P_ref = np.tile(p_ref, (len(y_true), 1))
    BS = brier_score(p_hat, y_true, classes)
    BS_ref = brier_score(P_ref, y_true, classes)
    return 1.0 - (BS / BS_ref if BS_ref > 0 else np.nan)

def predict_cluster_posteriors_CRF(
    data_groups,                 # list of arrays; data_groups[d] has shape (n_d, D)
    global_components,           # list-like; each has .logpdf(x) (Student-t or Gaussian)
    n_kd_total,                  # array (K, D_docs): counts N_{kd} per component k, doc d
    mk,                          # array (K,): table counts m_k
    alpha, gamma,                # scalars
    ignore_new=True              # 'closed menu' (no new component)
):
    """
    Returns:
      R_list: list over docs; each is (n_d, K) of responsibilities r_{ik} = P(k|x_i)
      idx_map: list of (doc_index, within_doc_index) for concatenation bookkeeping
    """
    K, D_docs = n_kd_total.shape
    m = float(np.sum(mk))
    R_list, idx_map = [], []

    for d in range(D_docs):
        n_kd = n_kd_total[:, d].astype(float)
        n_d  = float(n_kd.sum())
        # CRF closed-menu weights:
        numer = n_kd + alpha * (mk / (m + gamma))
        denom = n_d + alpha * (m / (m + gamma))
        pi_k  = numer / denom  # (K,)

        Xd = np.asarray(data_groups[d])
        n_d_pts = Xd.shape[0]
        R_d = np.empty((n_d_pts, K), dtype=float)


        log_pi = np.log(pi_k + 1e-300)
        for i, x in enumerate(Xd):
            log_fk = np.array([global_components[k].logpdf(x).item() for k in range(K)], dtype=float)
            # r_{ik} 鈭?pi_k * f_k(x)
            z = log_pi + log_fk
            z -= logsumexp(z)
            R_d[i, :] = np.exp(z)
            idx_map.append((d, i))
        R_list.append(R_d)

    return R_list, idx_map  # each R_d sums to 1 across K

def estimate_Q_c_given_k(R, y_true, classes=None, reg=1e-6):
    """
    Estimate Q[c,k] = P(class=c | cluster=k) from a labeled set.
    R: (N, K) responsibilities on those same N labeled points
    y_true: length-N integer/label array
    """
    if classes is None:
        classes = np.unique(y_true)
    C = len(classes)
    class_to_idx = {c:i for i,c in enumerate(classes)}
    y_idx = np.array([class_to_idx[y] for y in y_true], dtype=int)
    N, K = R.shape
    counts = np.zeros((C, K), dtype=float)
    for i in range(N):
        counts[y_idx[i], :] += R[i, :]
    counts += reg
    Q = counts / counts.sum(axis=0, keepdims=True)  # column-normalize over classes
    return Q, classes

def class_probs_from_R(R, Q):
    """Given R (N,K) and Q (C,K), return P_hat (N,C) with P(y=c|x)=sum_k Q[c,k] R[i,k]."""
    return R @ Q.T  # (N,K) @ (K,C)^T -> (N,C)

def brier_from_hdp_state(
    data_groups, global_components, n_kd_total, mk, alpha, gamma,
    y_true, labeled_mask=None, classes=None
):
    """
    - data_groups: list of groups/docs; concatenate for scoring
    - If you only want to use a subset to estimate Q (semi-supervised),
      pass labeled_mask (length N_all) marking which points are labeled.
    """
    R_list, idx_map = predict_cluster_posteriors_CRF(
        data_groups, global_components, n_kd_total, mk, alpha, gamma, ignore_new=True
    )
    R = np.vstack(R_list)  # (N_all, K)

    if labeled_mask is None:
        R_lab = R
        y_lab = y_true
    else:
        labeled_mask = np.asarray(labeled_mask, dtype=bool)
        R_lab = R[labeled_mask]
        y_lab = np.asarray(y_true)[labeled_mask]
    Q, classes = estimate_Q_c_given_k(R_lab, y_lab, classes=classes, reg=1e-6)

    P_hat = class_probs_from_R(R, Q)

    BS  = brier_score(P_hat, y_true, classes=classes)
    BSc = brier_by_class(P_hat, y_true, classes=classes)
    BSS = brier_skill_score(P_hat, y_true, classes=classes)
    return BS, BSc, BSS, P_hat, R, Q


# ECE


In [5]:
import numpy as np

def learn_Q_soft(R, y_true, classes=None, reg=1e-6):
    if classes is None:
        classes = np.unique(y_true)
    C, K = len(classes), R.shape[1]
    cls2idx = {c:i for i,c in enumerate(classes)}
    y_idx = np.array([cls2idx[y] for y in y_true], int)
    E = np.zeros((C, K), float)
    for i in range(R.shape[0]):        # expected counts per (class, cluster)
        E[y_idx[i]] += R[i]
    Q = (E + reg) / (E + reg).sum(axis=0, keepdims=True)
    return Q, classes

def class_probs_from_R(R, Q):
    return R @ Q.T  # (N,K) @ (K,C)^T -> (N,C)

def ece_toplabel(P_hat, y_true, n_bins=15):
    N, C = P_hat.shape
    conf = P_hat.max(axis=1)
    pred = P_hat.argmax(axis=1)
    correct = (pred == y_true).astype(float)
    bins = np.linspace(0.0, 1.0, n_bins + 1)
    idx  = np.clip(np.digitize(conf, bins) - 1, 0, n_bins - 1)
    ece = 0.0
    for b in range(n_bins):
        m = (idx == b)
        if not np.any(m): 
            continue
        acc_b  = correct[m].mean()
        conf_b = conf[m].mean()
        ece += m.mean() * abs(acc_b - conf_b)
    return ece

def ece_ovr(P_hat, y_true, n_bins=15, classes=None):
    P_hat = np.asarray(P_hat, float)
    N, C = P_hat.shape
    if classes is None:
        classes = np.arange(C)
    cls2idx = {c:i for i,c in enumerate(classes)}
    y_idx = np.array([cls2idx[y] for y in y_true], int)
    bins = np.linspace(0.0, 1.0, n_bins + 1)
    ece = 0.0
    for c in range(C):
        p_c = P_hat[:, c]
        idx = np.clip(np.digitize(p_c, bins) - 1, 0, n_bins - 1)
        for b in range(n_bins):
            m = (idx == b)
            if not np.any(m):
                continue
            acc_b  = (y_idx[m] == c).mean()
            conf_b = p_c[m].mean()
            ece += (m.sum() / N) * abs(acc_b - conf_b)
    return ece

def kfold_indices(N, k=5, shuffle=True, seed=0):
    rng = np.random.default_rng(seed)
    idx = np.arange(N)
    if shuffle:
        rng.shuffle(idx)
    return np.array_split(idx, k)

def probs_with_cv_Q(R, y_true, classes=None, kfold=5, reg=1e-6):
    """
    Learn Q on K-1 folds, predict P_hat on the held-out fold; repeat and stitch.
    R: (N,K) responsibilities from a fitted model (HDP or DPGMM)
    """
    N = R.shape[0]
    folds = kfold_indices(N, k=kfold, shuffle=True, seed=0)
    if classes is None:
        classes = np.unique(y_true)
    C = len(classes)
    P_hat = np.zeros((N, C), float)

    for val_idx in folds:
        train_idx = np.setdiff1d(np.arange(N), val_idx, assume_unique=True)
        Q, classes = learn_Q_soft(R[train_idx], y_true[train_idx], classes=classes, reg=reg)
        P_hat[val_idx] = class_probs_from_R(R[val_idx], Q)
    return P_hat, classes

def ece_ovr_classbalanced(P_hat, y_true, n_bins=15):
    N, C = P_hat.shape
    y_true = np.asarray(y_true)
    bins = np.linspace(0.0, 1.0, n_bins + 1)
    eces = []
    for c in range(C):
        p = P_hat[:, c]
        idx = np.clip(np.digitize(p, bins) - 1, 0, n_bins - 1)
        ece_c = 0.0
        for b in range(n_bins):
            m = (idx == b)
            if not m.any(): 
                continue
            acc_b  = (y_true[m] == c).mean()
            conf_b = p[m].mean()
            ece_c += (m.sum() / N) * abs(acc_b - conf_b)
        eces.append(ece_c)
    return float(np.mean(eces))

def compare_calibration_from_R(R_hdp, R_dpgmm, y_true, n_bins=15, kfold=5):
    # HDP
    P_hdp, classes = probs_with_cv_Q(R_hdp, y_true, kfold=kfold)
    ece_hdp_top = ece_toplabel(P_hdp, y_true, n_bins=n_bins)
    ece_hdp_ovr = ece_ovr(P_hdp, y_true, n_bins=n_bins, classes=classes)

    # DPGMM
    P_dpg, _ = probs_with_cv_Q(R_dpgmm, y_true, classes=classes, kfold=kfold)
    ece_dpg_top = ece_toplabel(P_dpg, y_true, n_bins=n_bins)
    ece_dpg_ovr = ece_ovr(P_dpg, y_true, n_bins=n_bins, classes=classes)

    return {
        "HDP": {"ECE_toplabel": ece_hdp_top, "ECE_OvR": ece_hdp_ovr, "P_hat": P_hdp, "classes": classes},
        "DPGMM": {"ECE_toplabel": ece_dpg_top, "ECE_OvR": ece_dpg_ovr, "P_hat": P_dpg, "classes": classes}
    }


# Five-seed experiments


In [6]:
def unsupervised_clustering_accuracy(y: Union[np.ndarray, torch.Tensor], y_pred: Union[np.ndarray, torch.Tensor]) -> tuple:
        """Unsupervised Clustering Accuracy
        """
        assert len(y_pred) == len(y)
        u = np.unique(y)
        n_true_clusters = len(u)
        v = np.unique(y_pred)
        n_pred_clusters = len(v)
        map_u = dict(zip(u, range(n_true_clusters)))
        map_v = dict(zip(v, range(n_pred_clusters)))
        inv_map_u = {v: k for k, v in map_u.items()}
        inv_map_v = {v: k for k, v in map_v.items()}
        r = np.zeros((n_pred_clusters, n_true_clusters), dtype=np.int64)
        for y_pred_, y_ in zip(y_pred, y):
            if y_ in map_u:
                r[map_v[y_pred_], map_u[y_]] += 1
        reward_matrix  = np.concatenate((r, r, r), axis=1)
        cost_matrix = reward_matrix.max() - reward_matrix
        row_assign, col_assign = linear_sum_assignment(cost_matrix)

        # Construct optimal assignments matrix
        row_assign = row_assign.reshape((-1, 1))  # (n,) to (n, 1) reshape
        col_assign = col_assign.reshape((-1, 1))  # (n,) to (n, 1) reshape
        assignments = np.concatenate((row_assign, col_assign), axis=1)
        assignments = [[inv_map_v[x], inv_map_u[y%n_true_clusters]] for x, y in assignments]

        optimal_reward = reward_matrix[row_assign, col_assign].sum() * 1.0
        return optimal_reward / y_pred.size, assignments  

def damage_detection_accuracy(k_dv, healthy_count=1500, healthy_reference_fraction=0.8, min_count=10):
    k_dv_int = np.asarray(k_dv).astype(int)
    healthy_reference_end = int(healthy_count * healthy_reference_fraction)
    health_labels_pre = np.unique(k_dv_int[:healthy_reference_end])
    counts = Counter(k_dv_int[:healthy_reference_end])
    health_labels = [label for label in health_labels_pre if counts[label] >= min_count]

    fn, fp = 0, 0
    for i in range(len(k_dv_int)):
        if i <= healthy_count and k_dv_int[i] not in health_labels:
            fp += 1
        elif i > healthy_count and k_dv_int[i] in health_labels:
            fn += 1

    dda = 1 - (fp + fn) / len(k_dv_int)
    return dda, fp, fn, health_labels

def build_global_components_from_sampler(sampler):
    global_components = {}
    component_id = 0
    for _, value in sampler.params.items():
        if value.nk > 0:
            global_components[component_id] = value
            component_id += 1
    return global_components

def run_crf_sampler_once(seed, X, y_true, hyperprior, iteration=200, snap_interval=200):
    np.random.seed(seed)
    torch.manual_seed(seed)

    pca_components = 10
    samples_per_segment = 100

    pca = PCA(n_components=pca_components)
    data_pca = pca.fit_transform(X).astype("float32")
    data = data_pca.reshape(-1, samples_per_segment, pca_components)

    sampler = GibbsSampler(snapshot_interval=20, compute_loglik=True)
    sampler.initialize(data=data)

    start_time = time.time()
    for tmp in range(int(iteration / snap_interval)):
        sampler.sample(
            snap_interval,
            concentration_parameter=None,
            hyper_prior=hyperprior,
        )
    sampler_runtime = time.time() - start_time

    sampler.create_k_dv(transform=True)
    k_dv = sampler._k_dv_array.astype(int)

    acc, assignments = unsupervised_clustering_accuracy(y_true, k_dv)
    dda, fp, fn, health_labels = damage_detection_accuracy(k_dv)

    global_components = build_global_components_from_sampler(sampler)
    BS, BSc, BSS, P_hat, R, Q = brier_from_hdp_state(
        data_groups=data,
        global_components=global_components,
        n_kd_total=sampler._n_kd,
        mk=sampler._m_k,
        alpha=sampler._alpha,
        gamma=sampler._gamma,
        y_true=y_true,
        labeled_mask=None,
        classes=None,
    )

    R_list, idx_map = predict_cluster_posteriors_CRF(
        data_groups=data,
        global_components=global_components,
        n_kd_total=sampler._n_kd,
        mk=sampler._m_k,
        alpha=sampler._alpha,
        gamma=sampler._gamma,
        ignore_new=True,
    )
    R_ece = np.vstack(R_list)
    P_hdp, classes = probs_with_cv_Q(R_ece, y_true, kfold=5)
    ece_hdp_top = ece_toplabel(P_hdp, y_true, n_bins=15)
    ece_hdp_ovr = ece_ovr(P_hdp, y_true, n_bins=15, classes=classes)
    ece_hdp_balanced = ece_ovr_classbalanced(P_hdp, y_true, n_bins=15)

    return {
        'seed': seed,
        'accuracy': acc,
        'damage_detection_accuracy': dda,
        'brier_score': BS,
        'brier_skill_score': BSS,
        'ece_toplabel': ece_hdp_top,
        'ece_ovr': ece_hdp_ovr,
        'ece_balanced': ece_hdp_balanced,
        'runtime_seconds': sampler_runtime,
        'num_components': sampler._K,
        'alpha': sampler._alpha,
        'gamma': sampler._gamma,
        'false_positive': fp,
        'false_negative': fn,
        'health_labels': health_labels,
        'assignments': assignments,
    }


# Experiment configuration


In [ ]:
experiment_seeds = [42, 1, 2, 3, 6]
iteration = 200
snap_interval = 50
hyper_parameter = {'alpha': 1.0, 'gamma': 1.0}
hyperprior = {'alpha_a': 1.0, 'alpha_b': 1.0, 'gamma_a': 5.0, 'gamma_b': 1.0}

y_true = y.numpy().astype(int)


# Run five-seed CRF HDP-GMM experiment


In [ ]:
seed_results = []

for run_id, seed in enumerate(experiment_seeds, start=1):
    print(f"\n===== Seed {seed} ({run_id}/{len(experiment_seeds)}) =====")
    result = run_crf_sampler_once(
        seed=seed,
        X=X,
        y_true=y_true,
        hyperprior=hyperprior,
        iteration=iteration,
        snap_interval=snap_interval,
    )
    seed_results.append(result)
    print(
        f"Seed {seed}: ACC={result['accuracy']:.6f}, "
        f"DDA={result['damage_detection_accuracy']:.6f}, "
        f"Brier={result['brier_score']:.6f}, "
        f"ECE_top={result['ece_toplabel']:.6f}, "
        f"runtime={result['runtime_seconds']:.2f}s, "
        f"K={result['num_components']}"
    )

results_df = pd.DataFrame(seed_results)
metric_columns = [
    'accuracy',
    'damage_detection_accuracy',
    'brier_score',
    'ece_balanced',
    'runtime_seconds',
]
summary_df = pd.DataFrame({
    'mean': results_df[metric_columns].mean(),
    'std': results_df[metric_columns].std(ddof=1),
})

print("\nPer-seed results")
display(results_df[[
    'seed',
    'accuracy',
    'damage_detection_accuracy',
    'brier_score',
    'ece_balanced',
    'runtime_seconds',
    'num_components',
    'alpha',
    'gamma',
]])

print("\nMean and sample standard deviation over seeds")
display(summary_df)
